<a href="https://colab.research.google.com/github/Carlosjara01/Inteligencia-Artificial/blob/main/Agentes_corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# 1 Instalaciones
!pip install -q pandas langchain langchain-community langchain-mistralai faiss-cpu sentence-transformers langchain_experimental langchain_text_splitters

In [16]:
# 2 Importaciones
import pandas as pd  # Importa pandas para leer y analizar el dataset

from google.colab import userdata  # Permite leer claves guardadas en Colab

from langchain_mistralai import ChatMistralAI  # Importa el modelo Mistral para LangChain

from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent  # Crea agente para consultar DataFrames

from langchain_community.embeddings import HuggingFaceEmbeddings  # Crea embeddings locales gratuitos

from langchain_community.vectorstores import FAISS  # Base vectorial para búsqueda semántica

from langchain_core.documents import Document  # Representa cada fila como documento

from langchain_text_splitters import RecursiveCharacterTextSplitter  # Divide textos largos

from langchain_core.prompts import ChatPromptTemplate  # Crea prompts estructurados

In [17]:
# 3 Cargar dataset
df = pd.read_csv("/content/dataset_normalizado.csv")  # Carga el archivo CSV normalizado

df.head()  # Muestra las primeras filas del dataset

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,0.021538,0.263736,0.941193,0.058824,0.175644,2/24/2003 0:00,Shipped,0.000000,0.090909,0.0,Motorcycles,0.342541,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,0.064615,0.307692,0.744940,0.235294,0.167916,5/7/2003 0:00,Shipped,0.333333,0.363636,0.0,Motorcycles,0.342541,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,0.104615,0.384615,0.928063,0.058824,0.250150,7/1/2003 0:00,Shipped,0.666667,0.545455,0.0,Motorcycles,0.342541,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,0.138462,0.428571,0.771061,0.294118,0.240030,8/25/2003 0:00,Shipped,0.666667,0.636364,0.0,Motorcycles,0.342541,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,0.181538,0.472527,1.000000,0.764706,0.347273,10/10/2003 0:00,Shipped,1.000000,0.818182,0.0,Motorcycles,0.342541,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [18]:
# 4 Ver información básica
print("Filas y columnas:", df.shape)  # Muestra cantidad de filas y columnas

print(df.columns)  # Muestra los nombres de las columnas

df.info()  # Muestra tipos de datos y valores nulos

Filas y columnas: (2823, 25)
Index(['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER',
       'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID',
       'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME',
       'DEALSIZE'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   float64
 1   QUANTITYORDERED   2823 non-null   float64
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   float64
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   object 
 6   STATUS            2823 non-null   object 
 7   QTR_ID            2823 non-null   float

In [19]:
# 5  Configurar API Key de Mistral
from google.colab import userdata  # Permite acceder a secretos de Colab

MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")  # Lee la API Key guardada

llm = ChatMistralAI(
    model="mistral-large-latest",  # Modelo Mistral recomendado para buena calidad
    temperature=0,  # Respuestas más precisas y menos creativas
    api_key=MISTRAL_API_KEY  # Clave de acceso a Mistral
)

In [20]:
# 6 Crear agente con Pandas
pandas_agent = create_pandas_dataframe_agent(
    llm,  # Modelo Mistral que responderá
    df,  # Dataset cargado con pandas
    verbose=True,  # Muestra el razonamiento técnico del agente
    allow_dangerous_code=True  # Necesario porque el agente ejecuta código sobre pandas
)

In [21]:
# 7 Convertir el dataset en corpus para RAG
documentos = []  # Lista donde guardaremos cada fila convertida en documento

for indice, fila in df.iterrows():  # Recorre cada fila del dataset
    texto = " | ".join([f"{columna}: {fila[columna]}" for columna in df.columns])  # Convierte la fila en texto
    documento = Document(
        page_content=texto,  # Texto que será usado por RAG
        metadata={"fila": indice}  # Guarda el número de fila como referencia
    )
    documentos.append(documento)  # Agrega el documento a la lista

In [22]:
# 8 Crear embeddings y FAISS
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # Modelo gratuito para convertir texto en vectores
)
vectorstore = FAISS.from_documents(
    documentos,  # Documentos creados desde el dataset
    embeddings  # Modelo que convierte texto en vectores
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# 9 Crear recuperador RAG
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}  # Recupera los 5 fragmentos más parecidos a la pregunta
)

In [24]:
# 10 Función RAG con Mistral
def responder_con_rag(pregunta):  # Define una función para responder usando RAG

    docs = retriever.invoke(pregunta)  # Busca documentos relacionados con la pregunta

    contexto = "\n\n".join([doc.page_content for doc in docs])  # Une los documentos recuperados

    prompt = ChatPromptTemplate.from_template("""
    Eres un asistente experto en análisis de datos.

    Responde en español, de forma clara y breve.

    Usa únicamente el siguiente contexto extraído del dataset:

    {contexto}

    Pregunta:
    {pregunta}
    """)  # Crea el prompt para Mistral

    chain = prompt | llm  # Une el prompt con el modelo Mistral

    respuesta = chain.invoke({
        "contexto": contexto,  # Envía el contexto recuperado
        "pregunta": pregunta  # Envía la pregunta del usuario
    })  # Ejecuta la cadena RAG

    return respuesta.content  # Devuelve solo el texto final

In [25]:
# 11 Probar RAG
respuesta = responder_con_rag(
    "¿Qué información existe sobre ventas pequeñas o Small?"
)  # Pregunta usando recuperación semántica

print(respuesta)  # Imprime la respuesta

En el dataset proporcionado, hay **2 registros con `DEALSIZE: Small`**:

1. **Orden 3** (ORDERNUMBER: 0.32307):
   - **Ventas (SALES)**: 0.0520 (valor normalizado).
   - **Producto**: Vintage Cars (S24_1937).
   - **Cliente**: Euro Shopping Channel (Madrid, España).
   - **Fecha**: 03/12/2003.

2. **Orden 5** (ORDERNUMBER: 0.79384):
   - **Ventas (SALES)**: 0.1430 (valor normalizado).
   - **Producto**: Classic Cars (S18_2238).
   - **Cliente**: Euro Shopping Channel (Madrid, España).
   - **Fecha**: 10/12/2004.

**Detalles comunes**:
- Ambos pedidos fueron **enviados (STATUS: Shipped)**.
- El cliente es el mismo en ambos casos.
- Los valores numéricos están normalizados (ej: `SALES` no representa el monto real).


In [26]:
# 12 Diferencia entre agente Pandas y RAG
pandas_agent.invoke(
    "Calcula el promedio de SALES por DEALSIZE."
)  # Ideal para cálculos con pandas

print(responder_con_rag(
    "Explícame qué registros relacionados a Classic Cars aparecen en el corpus."
))  # Ideal para buscar información textual



> Entering new AgentExecutor chain...


Thought: To calculate the average of SALES by DEALSIZE, I need to group the dataframe by the DEALSIZE column and then compute the mean of the SALES column for each group.
Action: python_repl_ast
Action Input:
df.groupby('DEALSIZE')['SALES'].mean()DEALSIZE
Large     0.574356
Medium    0.287949
Small     0.116138
Name: SALES, dtype: float64I now know the final answer.

Final Answer:
El promedio de SALES por DEALSIZE es el siguiente:
- Large: 0.574356
- Medium: 0.287949
- Small: 0.116138

> Finished chain.
En el corpus aparecen **4 registros relacionados con *Classic Cars***:

1. **Orden 0.316... (2003-12-02)**
   - **Ventas (SALES):** 0.574
   - **Cantidad (QUANTITYORDERED):** 0.462
   - **Precio unitario (PRICEEACH):** 1.0
   - **Tamaño del trato (DEALSIZE):** *Large*.

2. **Orden 0.784... (2004-12-07)**
   - **Ventas:** 0.198
   - **Cantidad:** 0.187
   - **Tamaño del trato:** *Medium*.

3. **Orden 0.012... (2003-01-31)**
   - **Ventas:** 0.219
   - **Cantidad:** 0.198
   - **Tamaño de